### 簡単な例

簡単な例で協調フィルタリングの振る舞いを確認する。

次元圧縮を用いて元の行列に戻す操作に以下が有名である。

- 特異値分解 (SVD)

 $X = U S V^T$ と分解する。
ここで対角行列であるSの非ゼロの要素を制限することで次元圧縮を行う。

- 非負値行列因子分解 (NMF)

 $X = W H$

非負行列W,Hを見つけて非負行列Xを構成する。なお、内部で乱数を用いる。


In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# 教材のため乱数を固定する。
np.random.seed(7)


def make_data(n1=10, n2=20):
    """make data

    Args:
        n1 (int, optional): size 1. Defaults to 10.
        n2 (int, optional): size 2. Defaults to 20.

    Returns:
        pd.DataFrame: data
    """
    X0 = np.zeros((n1, n2))

    for i in range(15):
        i1, i2 = np.random.randint(n1), np.random.randint(n2)
        X0[i1, i2] = 1

    X0[1, :] = 1
    X0[1, 4] = 0

    X0[2, :] = 1
    X0[2, 4:15] = 0

    return pd.DataFrame(X0)


g_df = make_data()
sns.heatmap(g_df.values)


'SVDによる寄与率'を確認しておく。

In [ ]:
def plot_svd_sdiag(X):
    """寄与率の表示

    Args:
        X (np.array): descriptor
    """
    u, sdiag, v = np.linalg.svd(X)
    if True:
        n = sdiag.shape[0]
        s = np.zeros((u.shape[1], v.shape[0]))
        s[:n, :n] = np.diag(sdiag)
        display(pd.DataFrame(s))
        u = np.matrix(u)
        v = np.matrix(v)  # = v.T
        s = np.matrix(s)
        usv = u*s*v
        """
        check matrixes are almost the same with np.allcose() 
        """
        print("check usv = original matrix? ", np.allclose(usv, X))

    sdiagsum = []
    for i in range(sdiag.shape[0]):
        sdiagsum.append(np.sum(sdiag[:i+1]))
    sdiagsum = np.array(sdiagsum)
    sdiag = sdiag / sdiagsum[-1]
    sdiagsum = sdiagsum / sdiagsum[-1]

    # 寄与率の表示
    fig, ax = plt.subplots()
    # plt.plot(np.log10(sdiag),"o-")
    ax.plot(sdiag, ".-", label="contribution")
    ax.plot(sdiagsum, ".-", label="comulative contribution")
    ax.set_ylabel("ratte")
    ax.set_xlabel("index")
    ax.legend()
    fig.show()


plot_svd_sdiag(g_df.values)


以下で次元圧縮と元行列のサイズへの構成を行う。

In [ ]:
from sklearn.decomposition import NMF


def make_recom_svd(df, nrank):
    """line up candicates by SVD

    Args:
        df (pd.DataFrame): data
        nrank (int): the maximum rank to reconstruct data

    Returns:
        pd.DataFrame: reconstruct data
    """
    X = df.values
    u, sdiag, v = np.linalg.svd(X)
    s = np.zeros((u.shape[1], v.shape[0]))
    s[:nrank, :nrank] = np.diag(sdiag[:nrank])
    display(pd.DataFrame(s))
    u = np.matrix(u)
    v = np.matrix(v)
    s = np.matrix(s)
    recom_svd = u * s * v
    return pd.DataFrame(recom_svd, index=df.index, columns=df.columns)


def make_recom_nmf(df, nrank):
    """line up candicates by NMF

    Args:
        df (pd.DataFrame): data
        nrank (int): the maximum rank to reconstruct data

    Returns:
        pd.DataFrame: reconstruct data
    """
    X = df.values
    model = NMF(n_components=nrank, init='random', random_state=1)
    W = model.fit_transform(X)
    H = model.components_
    W = np.matrix(W)
    H = np.matrix(H)
    WH = W*H
    """
    check how they are the same
    """
    if False:
        WHM = WH - X
        for i in range(WHM.shape[0]):
            for j in range(WHM.shape[1]):
                if np.abs(WHM[i, j]) > 0.1:
                    print(i, j, WHM[i, j])

    recom_nmf = WH
    return pd.DataFrame(recom_nmf, index=df.index, columns=df.columns)


def make_recom_correlation(df, nrank=None):
    """line up candicates by correlation

     X[material , structuretype]とすると
    ( X.T * X )[structuretype,structuretype] でstructuretype間の相関を与えるだろう。
    更にXをかけると[material, structuretype]の行列になる。
    recom = X[material , structuretype] * ( X.T * X )[structuretype,structuretype]
    Args:
        df (pd.DataFrame): data
        nrank (int): the maximum rank to reconstruct data

    Returns:
        pd.DataFrame: reconstruct data
    """
    # nrank はdummy
    X = np.matrix(df.values)

    recom = X * X.T * X
    # X^3のオーダーになっているので[0,1]に規格化する。
    vmax = recom.reshape(-1).max()
    vmin = recom.reshape(-1).min()
    recom = (recom - vmin)/(vmax-vmin)

    return pd.DataFrame(recom, index=df.index, columns=df.columns)


pythonの可視化関数を定義する。

In [ ]:
def plot_2df(df, df_transform, nrank):
    """plot the original data and reconstructed data

    Args:
        df (pd.DataFrame): data
        df_transform (pd.DataFrame): reconstructed data
        nrank (int): rank.
    """
    figsize = np.array(df.shape).astype(float)[::-1]*0.2
    figsize[0] = figsize[0]*1.8
    # (5*2,5*df_tranform.shape[0]/df_tranform.shape[1]))

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    ax = axes[0]
    sns.heatmap(df, ax=ax)
    ax = axes[1]
    ax.set_title("nrank={}".format(nrank))
    sns.heatmap(df_transform, ax=ax)
    plt.show()


1,3,5,7次元で動作を確認する。

In [ ]:
for _nrank in [1, 3, 5, 7]:
    g_df_tranform = make_recom_svd(g_df, _nrank)
    #df_tranform = make_recom_nmf(df,nrank)
    plot_2df(g_df, g_df_tranform, _nrank)


上の例でnrank=1の場合、つまり最初に、y=1でx4=0となる横ベクトルがあるという情報を得て、y=と似た横ベクトルであるy=2で適用して(2,4)=0,(2,5:14)=1と適用していると言える。
次元を増やしていくとその他の細かい構造を学んでいくと言える。

推薦システムは教師なし学習と呼ばれますが、
観測データを訓練データとテストデータに分割して、この場合はnrankがハイパーパラメタだとして最適化することもできます。
